# Baselines on Inspec: TF-IDF, YAKE and KeyBERT

TF-IDF, YAKE and KeyBERT run on the same 2,000 Inspec abstracts as the LLM, five phrases per document, no tuning on the gold labels, scored with the metrics of notebook 6 (Section IV-A3, Table 4).

Inputs: `dataset_inspec.csv`, `inspec_llama-3.1-8b-EN.csv`. Outputs in `results/e1_baselines/` (protocol details in its `README.md`). Gate: the six LLM means of notebook 6 must be reproduced within 5e-5 before any baseline runs. The embedding model is loaded offline from the local cache. Runtimes depend on the machine.

In [1]:
# ============================================================
# CONFIGURATION AND IMPORTS
# ============================================================
import os
os.environ["OMP_NUM_THREADS"] = "4"            # several notebooks run concurrently on one machine
# Inference uses the already cached embedding model, without network access.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import sys
sys.path.insert(0, "scripts")
import ast
import hashlib
import importlib.metadata
import inspect
import json
import logging
import platform
import random
import subprocess
import tempfile
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

import inspec_evaluation as ev

ROOT = Path.cwd()                              # the notebook runs from the repository root
OUT = ROOT / "results" / "e1_baselines"
SEED = 42
TOLERANCE = 0.00005  # half a unit at the reference's four-decimal precision
TOKEN_PATTERN = r"(?u)\b\w\w+\b"
VECTORIZER_CONFIG = {"ngram_range": (1, 4), "stop_words": "english", "min_df": 1,
                     "max_df": 1.0, "lowercase": True, "strip_accents": None,
                     "token_pattern": TOKEN_PATTERN, "analyzer": "word"}
BASELINE_CONFIG = {
    "target_k": 5, "gold_label_tuning": False,
    "tfidf": {**VECTORIZER_CONFIG, "norm": "l2", "use_idf": True,
              "smooth_idf": True, "sublinear_tf": False, "binary": False,
              "max_features": None, "dtype": "float64",
              "fit_scope": "all 2000 truncated insumo inputs; transductive",
              "ranking": "descending nonzero document TF-IDF; lexicographic phrase on exact ties",
              "deduplication": "unique vectorizer features; no fuzzy deduplication"},
    "yake": {"lan": "en", "n": 4, "dedup_lim": 0.9, "dedup_func": "seqm",
             "window_size": 1, "top": 5, "features": None, "lemmatize": False,
             "stopwords": "YAKE bundled English stopwords",
             "ranking": "native ascending score; stable candidate insertion order for ties",
             "top_k": "native top=5 after similarity deduplication; no padding"},
    "keybert": {**VECTORIZER_CONFIG, "model_name": ev.MODEL_NAME,
                "model_revision": ev.MODEL_REVISION, "top_n": 5,
                "use_mmr": False, "use_maxsum": False,
                "diversity": 0.5, "nr_candidates": 20,
                "diversity_settings_active": False, "seed_keywords": None,
                "llm": None, "embedding_batch_size": 128,
                "ranking": "native cosine argsort descending; NumPy default tie order; returned scores rounded to 4 decimals",
                "deduplication": "unique vectorizer features; no fuzzy deduplication",
                "fit_scope": "candidate vocabulary from all inputs; each document ranked only against its own candidates"},
    "output_policy": "keep native ranked top-five phrases, including post-cleaning collisions; never pad, drop documents, or alter saved LLM lists",
}
print("repository root:", ROOT)
print("results:", OUT.relative_to(ROOT))
print("evaluator:", ev.MODEL_NAME, "revision", ev.MODEL_REVISION, "| tau =", ev.TAU, "| batch size", ev.BATCH_SIZE)
print("reference 8B means (notebook 6):", ev.EXPECTED)

/home/mat/academic-writing/papers/from_text_to_structure/data_repo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


repository root: /home/mat/academic-writing/papers/from_text_to_structure/data_repo
results: results/e1_baselines
evaluator: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 revision e8f8c211226b894fcb81acc59f3b34ba3efd5f42 | tau = 0.7 | batch size 128
reference 8B means (notebook 6): {'jaccard_lex': 0.1422, 'soft_precision': 0.7939, 'soft_recall': 0.467, 'soft_f1': 0.5652, 'soft_mean_max': 0.7522, 'global_sem_sim': 0.8042}


In [2]:
# ============================================================
# HELPERS: HASHES, REFERENCE CHECK, DATA LOADING, EVALUATION
# ============================================================
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def write_json(path, value):
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + "\n")


def protected_hashes():
    """SHA-256 of the seven original notebooks and the four original data files (must not change)."""
    names = [*sorted(ROOT.glob("[1-7]. *.ipynb")),
             ROOT / "dataset_inspec.csv", ROOT / "inspec_llama-3.1-8b-EN.csv",
             ROOT / "EID_KEYWORDS.xlsx", ROOT / "human_eval_M1_8b.csv"]
    return {p.name: sha256(p) for p in names}


def check_reference_functions():
    """Compare syntax trees against notebook 6 (cells 3 and 4), not just expected summary values."""
    nb = json.loads((ROOT / "6. GROUND_TRUTH_INSPEC.ipynb").read_text())
    source = "\n".join("".join(nb["cells"][i]["source"]) for i in (3, 4))
    reference = ast.parse(source)
    module = ast.parse(inspect.getsource(ev))
    reference_nodes = [ast.dump(node, include_attributes=False) for node in reference.body]
    module_nodes = [ast.dump(node, include_attributes=False) for node in module.body]
    assert all(node in module_nodes for node in reference_nodes), "Reference evaluator drift"
    return {"reference_cells_zero_based": [3, 4], "ast_identical": True}


def load_data():
    gold = pd.read_csv(ROOT / "dataset_inspec.csv", dtype={"doc_id": str})
    gold["row_abs"] = np.arange(len(gold))
    llm = pd.read_csv(ROOT / "inspec_llama-3.1-8b-EN.csv")
    assert len(gold) == len(llm) == 2000
    assert gold.doc_id.nunique() == 2000
    assert llm.row_abs.is_unique and set(llm.row_abs) == set(gold.row_abs)
    assert gold[["doc_id", "insumo", "keywords_gt"]].notna().all().all()
    # This is the ONLY table passed to extraction functions: no gold labels.
    inputs = gold[["row_abs", "doc_id", "insumo"]].copy()
    inputs["input_chars_original"] = inputs.insumo.str.len()
    inputs["insumo"] = inputs.insumo.str.slice(stop=3000)
    inputs["input_chars_used"] = inputs.insumo.str.len()
    inputs["input_sha256"] = inputs.insumo.map(lambda s: hashlib.sha256(s.encode()).hexdigest())
    llm = gold[["row_abs", "doc_id"]].merge(llm, on="row_abs", validate="one_to_one", how="left")
    return gold, inputs, llm


def make_embedder(model):
    def embed(texts):
        return model.encode(texts, batch_size=ev.BATCH_SIZE, show_progress_bar=False,
                            convert_to_numpy=True, normalize_embeddings=False)
    return embed


def evaluate(gold, predictions, embed, method):
    assert predictions.row_abs.is_unique and set(predictions.row_abs) == set(gold.row_abs)
    joined = gold[["row_abs", "doc_id", "keywords_gt"]].merge(
        predictions[["row_abs", "keywords"]], on="row_abs", how="left", validate="one_to_one")
    # Missing field values follow notebook 6's safe_parse_list -> [] convention;
    # the row-identity check above prevents accidentally dropping observations.
    assert len(joined) == 2000
    records = []
    started = time.perf_counter()
    for row in joined.itertuples(index=False):
        gt = ev.clean_list(ev.safe_parse_list(row.keywords_gt), keep_null=False)
        pred = ev.clean_list(ev.safe_parse_list(row.keywords), keep_null=False)
        records.append({"method": method, "row_abs": row.row_abs, "doc_id": row.doc_id,
                        "gt_n": len(gt), "pred_n": len(pred),
                        "jaccard_lex": ev.jaccard(gt, pred),
                        **ev.soft_matching_metrics(gt, pred, embed, ev.TAU),
                        "global_sem_sim": ev.global_concat_similarity(gt, pred, embed)})
        if len(records) % 500 == 0:
            print(f"{method}: evaluated {len(records)}/2000 ({time.perf_counter()-started:.1f}s)", flush=True)
    result = pd.DataFrame(records)
    assert np.isfinite(result[ev.METRICS].to_numpy()).all()
    return result, time.perf_counter() - started


def summary(scores):
    return pd.DataFrame([{"method": method, "metric": metric, "n": len(group),
                          "mean": group[metric].mean(), "sd": group[metric].std(ddof=1)}
                         for method, group in scores.groupby("method", sort=False)
                         for metric in ev.METRICS])


def llm_predictions(llm):
    result = llm[["row_abs", "doc_id"]].copy()
    result["method"] = "llama_3.1_8b"
    # Preserve the saved string and list ordering; never enforce k on this file.
    result["keywords"] = llm.keywords_llm
    result["raw_n"] = result.keywords.map(lambda s: len(ev.safe_parse_list(s)))
    result["clean_n"] = result.keywords.map(lambda s: len(ev.clean_list(ev.safe_parse_list(s))))
    result["missing_prediction"] = llm.keywords_llm.isna()
    result["missing_raw_response"] = llm.raw_response.isna()
    result["status"] = np.where(result.missing_prediction, "missing_saved_prediction",
                                 np.where(result.raw_n == 0, "empty_saved_prediction", "saved_prediction"))
    result["error"] = ""
    result["extraction_seconds"] = np.nan  # historical runtime is unavailable
    return result


def prediction_record(row, method, ranked, seconds, error="", warning_text="", candidate_n=None):
    phrases = [str(phrase) for phrase, _ in ranked]
    clean = ev.clean_list(phrases)
    status = "failure" if error else ("empty_output" if not phrases else "ok")
    return {"method": method, "row_abs": int(row.row_abs), "doc_id": row.doc_id,
            "keywords": json.dumps(phrases, ensure_ascii=False),
            "scores": json.dumps([float(score) for _, score in ranked]),
            "raw_n": len(phrases), "clean_n": len(clean), "candidate_n": candidate_n,
            "status": status, "error": error, "warnings": warning_text,
            "missing_prediction": False, "missing_raw_response": None,
            "extraction_seconds": seconds}


class CaptureWarnings(logging.Handler):
    def __init__(self):
        super().__init__(logging.WARNING)
        self.messages = []

    def emit(self, record):
        self.messages.append(record.getMessage())


def summary_table(scores):
    """Means of the six metrics per method, one row per method (for display)."""
    return scores.groupby("method", sort=False)[ev.METRICS].mean()

In [3]:
# ============================================================
# BASELINE EXTRACTORS (input: IDs and truncated insumo text only)
# ============================================================
def extract_tfidf(inputs):
    from sklearn.feature_extraction.text import TfidfVectorizer
    started = time.perf_counter()
    vectorizer = TfidfVectorizer(**VECTORIZER_CONFIG, norm="l2", use_idf=True,
                                 smooth_idf=True, sublinear_tf=False, binary=False,
                                 max_features=None, dtype=np.float64)
    matrix = vectorizer.fit_transform(inputs.insumo.tolist())
    features = vectorizer.get_feature_names_out()
    fit_seconds = time.perf_counter() - started
    records = []
    for i, row in enumerate(inputs.itertuples(index=False)):
        begin = time.perf_counter()
        sparse = matrix.getrow(i)
        ranked = sorted(zip(features[sparse.indices], sparse.data), key=lambda pair: (-pair[1], pair[0]))[:5]
        records.append(prediction_record(row, "tfidf", ranked, time.perf_counter() - begin,
                                         candidate_n=sparse.nnz))
    return pd.DataFrame(records), {"extraction_seconds": time.perf_counter() - started,
                                    "fit_seconds": fit_seconds, "vocabulary_size": len(features),
                                    "per_document_timing": "ranking only; corpus fit reported separately"}


def extract_yake(inputs):
    import yake
    extractor = yake.KeywordExtractor(lan="en", n=4, dedup_lim=0.9, dedup_func="seqm",
                                      window_size=1, top=5, features=None, lemmatize=False)
    write_json(OUT / "yake_stopwords.json", sorted(extractor.stopword_set))
    records = []
    started = time.perf_counter()
    for row in inputs.itertuples(index=False):
        capture = CaptureWarnings()
        logging.getLogger().addHandler(capture)
        begin = time.perf_counter()
        error = ""
        ranked = []
        try:
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always")
                ranked = extractor.extract_keywords(row.insumo)
            capture.messages.extend(str(w.message) for w in caught)
            # YAKE catches exceptions internally and emits a warning before returning [].
            errors = [message for message in capture.messages if "Exception during keyword extraction" in message]
            error = " | ".join(errors)
        except Exception as exc:
            error = f"{type(exc).__name__}: {exc}"
        finally:
            logging.getLogger().removeHandler(capture)
        records.append(prediction_record(row, "yake", ranked, time.perf_counter() - begin,
                                         error, " | ".join(capture.messages)))
        if len(records) % 500 == 0:
            print(f"yake: extracted {len(records)}/2000", flush=True)
    return pd.DataFrame(records), {"extraction_seconds": time.perf_counter() - started,
                                    "per_document_timing": "full extraction, excluding extractor initialization"}


def extract_keybert(inputs, model):
    from keybert import KeyBERT
    from keybert.backend import BaseEmbedder
    from sklearn.feature_extraction.text import CountVectorizer

    class BatchedMiniLM(BaseEmbedder):
        """Standard sentence embeddings, batched for CPU memory and progress reporting."""
        def __init__(self):
            super().__init__(embedding_model=model)
            self.timings = []

        def embed(self, documents, verbose=False):
            begin = time.perf_counter()
            parts = []
            for start in range(0, len(documents), 4096):
                parts.append(model.encode(list(documents[start:start + 4096]), batch_size=128,
                                          show_progress_bar=False, convert_to_numpy=True,
                                          normalize_embeddings=False))
                done = min(start + 4096, len(documents))
                if done == len(documents) or (done // 4096) % 10 == 0:
                    print(f"keybert embeddings: {done}/{len(documents)}", flush=True)
            self.timings.append({"text_count": len(documents), "seconds": time.perf_counter() - begin})
            return np.concatenate(parts)

    started = time.perf_counter()
    backend = BatchedMiniLM()
    extractor = KeyBERT(model=backend)
    assert extractor.llm is None
    vectorizer = CountVectorizer(**VECTORIZER_CONFIG)
    ranked_lists = extractor.extract_keywords(inputs.insumo.tolist(), vectorizer=vectorizer,
                                              top_n=5, use_mmr=False, use_maxsum=False,
                                              diversity=0.5, nr_candidates=20, seed_keywords=None)
    if len(ranked_lists) != len(inputs):
        raise RuntimeError(f"KeyBERT returned {len(ranked_lists)} results for {len(inputs)} inputs")
    counts = vectorizer.transform(inputs.insumo.tolist()).getnnz(axis=1)
    records = []
    for row, ranked, candidate_n in zip(inputs.itertuples(index=False), ranked_lists, counts, strict=True):
        # Native KeyBERT catches ValueError and returns []; expose this if candidates existed.
        error = "KeyBERT returned empty output despite available candidates; possible internal ValueError" if candidate_n and not ranked else ""
        records.append(prediction_record(row, "keybert", ranked, None, error=error,
                                         candidate_n=int(candidate_n)))
    return pd.DataFrame(records), {"extraction_seconds": time.perf_counter() - started,
                                    "embedding_stages": backend.timings,
                                    "vocabulary_size": len(vectorizer.get_feature_names_out()),
                                    "per_document_timing": "unavailable: batched corpus extraction; no imputed per-document times"}


print("extractors defined: tfidf, yake, keybert")

extractors defined: tfidf, yake, keybert


In [4]:
# ============================================================
# RUN SETUP: PROTECTED HASHES, REFERENCE CHECK, SEEDS, DATA, ENVIRONMENT, MODEL
# ============================================================
branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=ROOT, text=True).strip()
OUT.mkdir(parents=True, exist_ok=True)
initial_hashes = protected_hashes()
reference_check = check_reference_functions()
print("reference evaluator check:", reference_check)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(4)
torch.use_deterministic_algorithms(True)
gold, inputs, llm = load_data()
print(f"documents: {len(inputs)}  |  inputs truncated at 3,000 chars: "
      f"{int((inputs.input_chars_original > 3000).sum())}  |  git branch: {branch}")
versions = {d.metadata["Name"]: d.version for d in importlib.metadata.distributions()}
write_json(OUT / "package_versions.json", dict(sorted(versions.items())))
write_json(OUT / "protected_artifact_hashes.json", initial_hashes)
print("key package versions:", {k: versions.get(k) for k in
      ["numpy", "pandas", "scikit-learn", "torch", "sentence-transformers", "transformers", "yake", "keybert"]})


def hardware_detail():
    """CPU model and physical memory from /proc (Linux); None where unavailable."""
    try:
        cpu = next(l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name"))
        mem_kib = next(int(l.split()[1]) for l in open("/proc/meminfo") if l.startswith("MemTotal"))
        return {"cpu_model": cpu, "memory_total_gib": round(mem_kib / 2 ** 20, 1)}
    except Exception:
        return None


hardware = hardware_detail()
metadata = {"started_utc": datetime.now(timezone.utc).isoformat(), "branch": branch,
            "seed": SEED, "python": platform.python_version(), "platform": platform.platform(),
            "machine": platform.machine(), "cpu_count": os.cpu_count(), "torch_threads": 4,
            "device": "cpu", "model_name": ev.MODEL_NAME, "model_revision": ev.MODEL_REVISION,
            "tau": ev.TAU, "batch_size": ev.BATCH_SIZE, "sd_ddof": 1,
            "input_count": len(inputs), "input_truncation_chars": 3000,
            "truncated_document_ids": inputs.loc[inputs.input_chars_original > 3000, ["row_abs", "doc_id"]].to_dict("records"),
            "paid_api_calls": 0, "historical_llm_runtime_seconds": None,
            "historical_llm_cost": None, "reference_check": reference_check}
metadata["llm_validation_reused"] = False
metadata["baseline_configurations"] = BASELINE_CONFIG
metadata["hardware_detail"] = hardware
metadata["hardware_detail_unavailable"] = None if hardware else "CPU model and physical RAM could not be read from /proc; platform and logical CPU count measured"
inputs.drop(columns="insumo").to_csv(OUT / "input_manifest.csv", index=False)
started = time.perf_counter()
model = SentenceTransformer(ev.MODEL_NAME, revision=ev.MODEL_REVISION,
                            device="cpu", local_files_only=True)
metadata["model_load_seconds"] = time.perf_counter() - started
metadata["model_max_seq_length"] = model.max_seq_length
metadata["model_dtype"] = str(next(model.parameters()).dtype)
write_json(OUT / "metadata.json", metadata)
write_json(OUT / "configuration.json", {"seed": SEED, "input_column": "insumo",
                                       "truncate_chars": 3000,
                                       "evaluation": {"model": ev.MODEL_NAME, "revision": ev.MODEL_REVISION,
                                                      "tau": ev.TAU, "batch_size": ev.BATCH_SIZE,
                                                      "keep_null": False, "ddof": 1,
                                                      "normalize_embeddings": False,
                                                      "reference_function_cells": [3, 4]},
                                       "baselines": BASELINE_CONFIG})
embed = make_embedder(model)
print(json.dumps({k: metadata[k] for k in ("python", "platform", "machine", "cpu_count", "torch_threads",
                                            "hardware_detail", "model_load_seconds", "model_max_seq_length",
                                            "model_dtype", "truncated_document_ids")}, indent=2))

reference evaluator check: {'reference_cells_zero_based': [3, 4], 'ast_identical': True}


documents: 2000  |  inputs truncated at 3,000 chars: 1  |  git branch: ieee


key package versions: {'numpy': '2.1.3', 'pandas': '2.2.3', 'scikit-learn': '1.9.0', 'torch': '2.8.0+cpu', 'sentence-transformers': '5.1.1', 'transformers': '4.57.6', 'yake': '0.7.3', 'keybert': '0.9.0'}


{
  "python": "3.12.3",
  "platform": "Linux-7.0.0-31-generic-x86_64-with-glibc2.39",
  "machine": "x86_64",
  "cpu_count": 16,
  "torch_threads": 4,
  "hardware_detail": {
    "cpu_model": "AMD Ryzen 7 7735U with Radeon Graphics",
    "memory_total_gib": 46.5
  },
  "model_load_seconds": 1.8898122459941078,
  "model_max_seq_length": 128,
  "model_dtype": "torch.float32",
  "truncated_document_ids": [
    {
      "row_abs": 164,
      "doc_id": "1150"
    }
  ]
}


In [5]:
# ============================================================
# GATE: REPRODUCE THE PUBLISHED LLM SCORES (notebook 6) WITHIN 5e-5
# ============================================================
preds = llm_predictions(llm)
preds.to_csv(OUT / "predictions_llama_3.1_8b.csv", index=False)
scores, elapsed = evaluate(gold, preds, embed, "llama_3.1_8b")
scores.to_csv(OUT / "metrics_llama_3.1_8b.csv", index=False)
checks = []
for metric in ev.METRICS:
    actual = float(scores[metric].mean())
    checks.append({"metric": metric, "expected": ev.EXPECTED[metric], "reproduced": actual,
                   "difference": actual - ev.EXPECTED[metric],
                   "passed": abs(actual - ev.EXPECTED[metric]) <= TOLERANCE})
validation = {"checks": checks, "absolute_tolerance": TOLERANCE,
              "passed": all(c["passed"] for c in checks),
              "evaluation_seconds": elapsed, **reference_check}
validation["artifact_sha256"] = {str(p.relative_to(ROOT)): sha256(p) for p in
                                  [ROOT / "scripts/inspec_evaluation.py",
                                   OUT / "predictions_llama_3.1_8b.csv", OUT / "metrics_llama_3.1_8b.csv"]}
write_json(OUT / "llm_validation.json", validation)
summary(scores).to_csv(OUT / "summary.csv", index=False)
print("\nLLM GATE:", "PASS" if validation["passed"] else "FAIL", f"(tolerance {TOLERANCE}, evaluation {elapsed:.1f} s)")
print(pd.DataFrame(checks).to_string(index=False))
assert protected_hashes() == initial_hashes, "Protected input changed"
if not validation["passed"]:
    raise RuntimeError("LLM reproduction differs from saved reference; STOP before baseline execution")

llama_3.1_8b: evaluated 500/2000 (438.5s)


llama_3.1_8b: evaluated 1000/2000 (1182.5s)


llama_3.1_8b: evaluated 1500/2000 (1720.6s)


llama_3.1_8b: evaluated 2000/2000 (2142.3s)



LLM GATE: PASS (tolerance 5e-05, evaluation 2142.3 s)
        metric  expected  reproduced    difference  passed
   jaccard_lex    0.1422    0.142228  2.759743e-05    True
soft_precision    0.7939    0.793900  0.000000e+00    True
   soft_recall    0.4670    0.466999 -8.972089e-07    True
       soft_f1    0.5652    0.565183 -1.687299e-05    True
 soft_mean_max    0.7522    0.752155 -4.542584e-05    True
global_sem_sim    0.8042    0.804198 -2.066237e-06    True


In [6]:
# ============================================================
# BASELINES: EXTRACT AND EVALUATE TF-IDF, YAKE AND KEYBERT
# ============================================================
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
write_json(OUT / "sklearn_english_stopwords.json", sorted(ENGLISH_STOP_WORDS))
all_predictions, all_scores = [preds], [scores]
runtimes = [{"method": "llama_3.1_8b", "extraction_seconds": None,
             "evaluation_seconds": elapsed, "extraction_source": "historical; unavailable",
             "evaluation_source": "independent local reproduction"}]
metadata["method_details"] = {}
for method, extractor in [("tfidf", extract_tfidf), ("yake", extract_yake),
                          ("keybert", lambda frame: extract_keybert(frame, model))]:
    print(f"\nStarting {method} extraction (no gold columns in extractor input)", flush=True)
    method_predictions, detail = extractor(inputs)
    assert len(method_predictions) == 2000 and method_predictions.row_abs.is_unique
    method_predictions.to_csv(OUT / f"predictions_{method}.csv", index=False)
    method_scores, eval_seconds = evaluate(gold, method_predictions, embed, method)
    method_scores.to_csv(OUT / f"metrics_{method}.csv", index=False)
    all_predictions.append(method_predictions)
    all_scores.append(method_scores)
    runtimes.append({"method": method, "extraction_seconds": detail["extraction_seconds"],
                     "evaluation_seconds": eval_seconds, "extraction_source": "measured locally",
                     "evaluation_source": "measured locally"})
    metadata["method_details"][method] = detail
    write_json(OUT / "metadata.json", metadata)
    print(f"Completed {method}: extraction={detail['extraction_seconds']:.2f}s evaluation={eval_seconds:.2f}s", flush=True)
    print(f"example ({method}, row_abs 0): {method_predictions.keywords.iloc[0]}")

print("\n=== Means over the 2,000 documents ===")
print(summary_table(pd.concat(all_scores, ignore_index=True)).round(4).to_string())


Starting tfidf extraction (no gold columns in extractor input)


tfidf: evaluated 500/2000 (436.8s)


tfidf: evaluated 1000/2000 (536.2s)


tfidf: evaluated 1500/2000 (629.8s)


tfidf: evaluated 2000/2000 (721.1s)


...

keybert: evaluated 1000/2000 (113.5s)


keybert: evaluated 1500/2000 (167.8s)


keybert: evaluated 2000/2000 (226.5s)


Completed keybert: extraction=1510.28s evaluation=226.51s


example (keybert, row_abs 0): ["separate account industry", "separate account industry supplying", "separate accounts mainstream investment", "shaking separate account industry", "separate accounts mainstream"]

=== Means over the 2,000 documents ===
              jaccard_lex  soft_precision  soft_recall  soft_f1  soft_mean_max  global_sem_sim
method                                                                                        
llama_3.1_8b       0.1422          0.7939       0.4670   0.5652         0.7522          0.8042
tfidf              0.0469          0.6211       0.2777   0.3631         0.6389          0.6623
yake               0.0496          0.6789       0.2416   0.3328         0.6360          0.6921
keybert            0.0119          0.7941       0.2026   0.3034         0.6411          0.7325


In [7]:
# ============================================================
# AGGREGATE OUTPUTS: predictions_all, metrics_all, summary, distributions, quality, runtimes, comparison.md
# ============================================================
def save_aggregate_outputs(predictions, scores, runtimes):
    all_predictions = pd.concat(predictions, ignore_index=True)
    all_scores = pd.concat(scores, ignore_index=True)
    assert len(all_predictions) == len(all_scores) == 8000
    assert not all_predictions.duplicated(["method", "row_abs"]).any()
    assert not all_scores.duplicated(["method", "row_abs"]).any()
    all_predictions.to_csv(OUT / "predictions_all.csv", index=False)
    all_scores.to_csv(OUT / "metrics_all.csv", index=False)
    table = summary(all_scores)
    table.to_csv(OUT / "summary.csv", index=False)
    distributions = []
    for field in ("raw_n", "clean_n"):
        dist = all_predictions.groupby(["method", field], dropna=False).size().reset_index(name="documents")
        dist = dist.rename(columns={field: "phrase_count"})
        dist["count_type"] = field
        distributions.append(dist)
    pd.concat(distributions, ignore_index=True).to_csv(OUT / "output_length_distributions.csv", index=False)
    quality = []
    for method, group in all_predictions.groupby("method", sort=False):
        method_scores = all_scores[all_scores.method == method]
        quality.append({"method": method, "documents": len(group),
                        "raw_empty": int((group.raw_n == 0).sum()),
                        "clean_empty": int((group.clean_n == 0).sum()),
                        "raw_not_five": int((group.raw_n != 5).sum()),
                        "clean_not_five": int((group.clean_n != 5).sum()),
                        "reported_failures": int(group.error.fillna("").ne("").sum()),
                        "historical_failure_status": "unavailable" if method == "llama_3.1_8b" else "not_applicable",
                        "missing_predictions": int(group.missing_prediction.fillna(False).sum()),
                        "missing_raw_responses": int(group.missing_raw_response.fillna(False).sum()) if method == "llama_3.1_8b" else None,
                        "warning_documents": int(group.warnings.fillna("").ne("").sum()),
                        "missing_metric_values": int(method_scores[ev.METRICS].isna().sum().sum())})
    pd.DataFrame(quality).to_csv(OUT / "quality_counts.csv", index=False)
    pd.DataFrame(runtimes).to_csv(OUT / "runtimes.csv", index=False)
    lines = ["# E1 extraction baseline results", "", "All scores are document means ± sample SD (ddof=1), n=2,000 per method.", "",
             "| Method | " + " | ".join(ev.METRICS) + " |",
             "|---|" + "---|" * len(ev.METRICS)]
    for method in all_scores.method.unique():
        rows = table[table.method == method].set_index("metric")
        lines.append("| " + method + " | " + " | ".join(
            f"{rows.loc[m, 'mean']:.4f} ± {rows.loc[m, 'sd']:.4f}" for m in ev.METRICS) + " |")
    lines += ["", "See README.md for configurations, timing scope, and methodological limitations.", ""]
    (OUT / "comparison.md").write_text("\n".join(lines))
    return all_predictions, all_scores, table


predictions_all, scores_all, summary_long = save_aggregate_outputs(all_predictions, all_scores, runtimes)
assert protected_hashes() == initial_hashes, "Protected input changed"
metadata["completed_utc"] = datetime.now(timezone.utc).isoformat()
metadata["status"] = "complete"
metadata["protected_artifacts_unchanged"] = True
metadata["source_sha256"] = {str(p.relative_to(ROOT)): sha256(p) for p in sorted((ROOT / "scripts").glob("*.py"))}
write_json(OUT / "metadata.json", metadata)
print((OUT / "comparison.md").read_text())
print("=== quality_counts.csv ===")
print(pd.read_csv(OUT / "quality_counts.csv").to_string(index=False))
print("\n=== runtimes.csv (seconds, hardware dependent) ===")
print(pd.read_csv(OUT / "runtimes.csv").to_string(index=False))
print("\n=== output_length_distributions.csv (cleaned counts) ===")
dist = pd.read_csv(OUT / "output_length_distributions.csv")
print(dist[dist.count_type == "clean_n"].pivot(index="method", columns="phrase_count", values="documents").fillna(0).astype(int).to_string())

# E1 extraction baseline results

All scores are document means ± sample SD (ddof=1), n=2,000 per method.

| Method | jaccard_lex | soft_precision | soft_recall | soft_f1 | soft_mean_max | global_sem_sim |
|---|---|---|---|---|---|---|
| llama_3.1_8b | 0.1422 ± 0.1118 | 0.7939 ± 0.2069 | 0.4670 ± 0.1799 | 0.5652 ± 0.1626 | 0.7522 ± 0.0791 | 0.8042 ± 0.0785 |
| tfidf | 0.0469 ± 0.0588 | 0.6211 ± 0.2549 | 0.2777 ± 0.1604 | 0.3631 ± 0.1734 | 0.6389 ± 0.0897 | 0.6623 ± 0.1250 |
| yake | 0.0496 ± 0.0580 | 0.6789 ± 0.2798 | 0.2416 ± 0.1564 | 0.3328 ± 0.1738 | 0.6360 ± 0.0989 | 0.6921 ± 0.1269 |
| keybert | 0.0119 ± 0.0287 | 0.7941 ± 0.2744 | 0.2026 ± 0.1330 | 0.3034 ± 0.1623 | 0.6411 ± 0.0718 | 0.7325 ± 0.0991 |

See README.md for configurations, timing scope, and methodological limitations.

=== quality_counts.csv ===
      method  documents  raw_empty  clean_empty  raw_not_five  clean_not_five  reported_failures historical_failure_status  missing_predictions  missing_raw_responses  warning

      method  extraction_seconds  evaluation_seconds       extraction_source              evaluation_source
llama_3.1_8b                 NaN         2142.297076 historical; unavailable independent local reproduction
       tfidf            4.356212          721.068929        measured locally               measured locally
        yake           25.035700          333.091784        measured locally               measured locally
     keybert         1510.282603          226.511370        measured locally               measured locally

=== output_length_distributions.csv (cleaned counts) ===
phrase_count  0   4     5  6
method                      
keybert       0   0  2000  0
llama_3.1_8b  1  19  1979  1
tfidf         0   0  2000  0
yake          0   1  1999  0


In [8]:
# ============================================================
# ADDITIONAL CHECKS: YAKE REPEATABILITY, KEYBERT NATIVE BACKEND, TOKEN LENGTHS
# ============================================================
# 1. YAKE determinism in an independent process (same configuration, same truncated inputs).
yake_code = r"""
import json, sys, warnings, logging
warnings.simplefilter("ignore"); logging.disable(logging.WARNING)
import pandas as pd, yake
gold = pd.read_csv("dataset_inspec.csv", dtype={"doc_id": str})
texts = gold.insumo.str.slice(stop=3000).tolist()
ex = yake.KeywordExtractor(lan="en", n=4, dedup_lim=0.9, dedup_func="seqm", window_size=1, top=5, features=None, lemmatize=False)
out = [[[str(p), float(s)] for p, s in ex.extract_keywords(t)] for t in texts]
json.dump(out, open(sys.argv[1], "w"), ensure_ascii=False)
"""
started = time.perf_counter()
fd, tmp_path = tempfile.mkstemp(suffix="_yake_repeat.json"); os.close(fd)
subprocess.run([sys.executable, "-c", yake_code, tmp_path], cwd=ROOT, check=True)
repeat = json.load(open(tmp_path)); os.remove(tmp_path)
yake_saved = pd.read_csv(OUT / "predictions_yake.csv", dtype={"doc_id": str}).sort_values("row_abs")
mismatches = [int(r) for r, kw, sc, rep in zip(yake_saved.row_abs, yake_saved.keywords, yake_saved.scores, repeat)
              if [json.loads(kw), json.loads(sc)] != [[p for p, _ in rep], [s for _, s in rep]]]
yake_check = {"method": "yake", "documents_checked": int(len(yake_saved)),
              "ordered_phrase_and_score_mismatches": mismatches, "seconds": time.perf_counter() - started,
              "purpose": "independent-process determinism check; predictions and evaluation results unchanged"}
write_json(OUT / "yake_repeatability_check.json", yake_check)
print("YAKE repeatability (independent process):", "identical" if not mismatches else f"{len(mismatches)} mismatches",
      f"({yake_check['seconds']:.1f} s)")

# 2. KeyBERT: fixed rows against the unmodified native SentenceTransformer backend (no batching wrapper).
from keybert import KeyBERT
from sklearn.feature_extraction.text import CountVectorizer
native = KeyBERT(model=model)
kb_saved = pd.read_csv(OUT / "predictions_keybert.csv", dtype={"doc_id": str}).set_index("row_abs")
kb_checks = []
for r in (0, 999, 1999):
    ranked = native.extract_keywords(inputs.insumo.iloc[r], vectorizer=CountVectorizer(**VECTORIZER_CONFIG),
                                     top_n=5, use_mmr=False, use_maxsum=False, diversity=0.5,
                                     nr_candidates=20, seed_keywords=None)
    saved_phrases, saved_scores = json.loads(kb_saved.keywords[r]), json.loads(kb_saved.scores[r])
    kb_checks.append({"row_abs": r, "ordered_phrases_identical": [p for p, _ in ranked] == saved_phrases,
                      "maximum_score_difference": float(max([abs(float(s) - t) for (_, s), t in zip(ranked, saved_scores)] or [0.0]))})
kb_check = {"purpose": "fixed sample check against unmodified native SentenceTransformer KeyBERT backend; no tuning",
            "checks": kb_checks, "passed": all(c["ordered_phrases_identical"] and c["maximum_score_difference"] == 0.0 for c in kb_checks)}
write_json(OUT / "keybert_native_check.json", kb_check)
print("KeyBERT native-backend check:", "PASS" if kb_check["passed"] else "FAIL", kb_checks)

# 3. Token lengths of the untruncated inputs and gold concatenations against the 128-token model limit (diagnostic only).
def token_lengths(texts):
    return [len(ids) for ids in model.tokenizer(list(texts), add_special_tokens=True, truncation=False)["input_ids"]]
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    len_docs = token_lengths(inputs.insumo)
    len_gold = token_lengths([" ; ".join(ev.clean_list(ev.safe_parse_list(s))) for s in gold.keywords_gt])
token_diag = {"model_revision": ev.MODEL_REVISION, "max_seq_length": int(model.max_seq_length),
              "counts_include_special_tokens": True,
              "truncated_insumo": {"documents": len(len_docs), "over_128_tokens": int(sum(l > 128 for l in len_docs)),
                                   "maximum_tokens": int(max(len_docs))},
              "gold_concatenations": {"documents": len(len_gold), "over_128_tokens": int(sum(l > 128 for l in len_gold)),
                                      "maximum_tokens": int(max(len_gold))}}
write_json(OUT / "token_length_diagnostics.json", token_diag)
print("token length diagnostics:", json.dumps(token_diag))

YAKE repeatability (independent process): identical (16.4 s)


KeyBERT native-backend check: PASS [{'row_abs': 0, 'ordered_phrases_identical': True, 'maximum_score_difference': 0.0}, {'row_abs': 999, 'ordered_phrases_identical': True, 'maximum_score_difference': 0.0}, {'row_abs': 1999, 'ordered_phrases_identical': True, 'maximum_score_difference': 0.0}]


Token indices sequence length is longer than the specified maximum sequence length for this model (190 > 128). Running this sequence through the model will result in indexing errors


token length diagnostics: {"model_revision": "e8f8c211226b894fcb81acc59f3b34ba3efd5f42", "max_seq_length": 128, "counts_include_special_tokens": true, "truncated_insumo": {"documents": 2000, "over_128_tokens": 1429, "maximum_tokens": 719}, "gold_concatenations": {"documents": 2000, "over_128_tokens": 110, "maximum_tokens": 239}}


In [9]:
# ============================================================
# VERIFY THE COMPLETED ARTIFACTS (read back and recheck; writes artifact_validation.json)
# ============================================================
def verify_completed_artifacts():
    """Read back results and independently check identities, counts and summaries."""
    check_reference_functions()
    metadata = json.loads((OUT / "metadata.json").read_text())
    assert metadata["status"] == "complete"
    assert protected_hashes() == json.loads((OUT / "protected_artifact_hashes.json").read_text())
    for name, checksum in metadata["source_sha256"].items():
        assert sha256(ROOT / name) == checksum, f"Source changed since run: {name}"
    gold, inputs, llm = load_data()
    predictions = pd.read_csv(OUT / "predictions_all.csv", dtype={"doc_id": str})
    scores = pd.read_csv(OUT / "metrics_all.csv", dtype={"doc_id": str})
    methods = {"llama_3.1_8b", "tfidf", "yake", "keybert"}
    assert set(predictions.method) == set(scores.method) == methods
    assert len(predictions) == len(scores) == 8000
    assert not predictions.duplicated(["method", "row_abs"]).any()
    assert not scores.duplicated(["method", "row_abs"]).any()
    assert np.isfinite(scores[ev.METRICS].to_numpy()).all()
    gold_lists = [ev.clean_list(ev.safe_parse_list(s)) for s in gold.keywords_gt]
    for method in sorted(methods):
        p = predictions[predictions.method == method].sort_values("row_abs").reset_index(drop=True)
        s = scores[scores.method == method].sort_values("row_abs").reset_index(drop=True)
        assert len(p) == len(s) == 2000
        assert p.row_abs.tolist() == s.row_abs.tolist() == list(range(2000))
        assert p.doc_id.tolist() == s.doc_id.tolist() == gold.doc_id.tolist()
        raw = [ev.safe_parse_list(value) for value in p.keywords]
        clean = [ev.clean_list(value) for value in raw]
        assert p.raw_n.tolist() == [len(value) for value in raw]
        assert p.clean_n.tolist() == s.pred_n.tolist() == [len(value) for value in clean]
        assert s.gt_n.tolist() == [len(value) for value in gold_lists]
        lexical = [ev.jaccard(g, candidate) for g, candidate in zip(gold_lists, clean, strict=True)]
        np.testing.assert_allclose(s.jaccard_lex, lexical, rtol=0, atol=1e-15)
        single = pd.read_csv(OUT / f"metrics_{method}.csv", dtype={"doc_id": str})
        pd.testing.assert_frame_equal(s, single.reset_index(drop=True), check_exact=False, atol=1e-15, rtol=0)
        if method == "llama_3.1_8b":
            assert p.keywords.tolist() == llm.keywords_llm.tolist(), "Saved LLM predictions altered"
    expected_summary = summary(scores).sort_values(["method", "metric"]).reset_index(drop=True)
    saved_summary = pd.read_csv(OUT / "summary.csv").sort_values(["method", "metric"]).reset_index(drop=True)
    pd.testing.assert_frame_equal(expected_summary, saved_summary, check_exact=False, atol=1e-15, rtol=0)
    manifest = pd.read_csv(OUT / "input_manifest.csv", dtype={"doc_id": str})
    pd.testing.assert_frame_equal(manifest, inputs.drop(columns="insumo"), check_dtype=False)
    validation = json.loads((OUT / "llm_validation.json").read_text())
    assert validation["passed"]
    for name, checksum in validation["artifact_sha256"].items():
        assert sha256(ROOT / name) == checksum
    report = {"passed": True, "methods": sorted(methods), "prediction_rows": len(predictions),
              "metric_rows": len(scores), "metric_values": len(scores) * len(ev.METRICS),
              "missing_metric_values": 0, "row_identity_and_counts_checked": True,
              "saved_llm_predictions_identical": True, "lexical_scores_recomputed": True,
              "summary_means_and_sample_sds_recomputed": True,
              "protected_artifacts_unchanged": True, "input_manifest_verified": True,
              "verified_utc": datetime.now(timezone.utc).isoformat()}
    write_json(OUT / "artifact_validation.json", report)
    return report


print(json.dumps(verify_completed_artifacts(), indent=2))

{
  "passed": true,
  "methods": [
    "keybert",
    "llama_3.1_8b",
    "tfidf",
    "yake"
  ],
  "prediction_rows": 8000,
  "metric_rows": 8000,
  "metric_values": 48000,
  "missing_metric_values": 0,
  "row_identity_and_counts_checked": true,
  "saved_llm_predictions_identical": true,
  "lexical_scores_recomputed": true,
  "summary_means_and_sample_sds_recomputed": true,
  "protected_artifacts_unchanged": true,
  "input_manifest_verified": true,
  "verified_utc": "2026-09-24T05:09:32.252767+00:00"
}


## Check against the manuscript

In [10]:
# ============================================================
# CHECK AGAINST THE MANUSCRIPT (values transcribed from main.tex: Table 4, Sections IV-A2, IV-A3, V-B)
# ============================================================
def fmt_like(manuscript: str, value) -> str:
    """Format `value` with the precision used in the manuscript string (decimals or percent)."""
    s = manuscript.replace(",", "")
    if s.endswith("%"):
        dec = len(s[:-1].split(".")[1]) if "." in s else 0
        return f"{value * 100:.{dec}f}%"
    dec = len(s.split(".")[1]) if "." in s else 0
    return f"{int(round(value)):,}" if dec == 0 else f"{value:.{dec}f}"

means = summary_long.pivot(index="method", columns="metric", values="mean")
TABLE4 = {  # Table 4 of main.tex: Jaccard, Soft P, Soft R, Soft F1, SMM, Global
    "llama_3.1_8b": ("LLaMA 3.1 8B", ["0.142", "0.794", "0.467", "0.565", "0.752", "0.804"]),
    "tfidf":        ("TF-IDF",       ["0.047", "0.621", "0.278", "0.363", "0.639", "0.662"]),
    "yake":         ("YAKE",         ["0.050", "0.679", "0.242", "0.333", "0.636", "0.692"]),
    "keybert":      ("KeyBERT",      ["0.012", "0.794", "0.203", "0.303", "0.641", "0.732"]),
}
COLS = ["jaccard_lex", "soft_precision", "soft_recall", "soft_f1", "soft_mean_max", "global_sem_sim"]
LABELS = ["Jaccard", "Soft P", "Soft R", "Soft F1", "SMM", "Global"]
rows = []
for method, (label, values) in TABLE4.items():
    for col, name, v in zip(COLS, LABELS, values):
        rows.append(("Table 4 (Sec. IV-A3)", f"{label}: {name}", v, float(means.loc[method, col])))
for col, name, v in zip(COLS, LABELS, ["0.14", "0.79", "0.47", "0.57", "0.75", "0.80"]):
    rows.append(("Sec. IV-A2 text", f"LLaMA 3.1 8B: {name}", v, float(means.loc["llama_3.1_8b", col])))
rows.append(("Sec. IV-A3 text", "KeyBERT soft precision vs LLM (0.794 against 0.794)", "0.794", float(means.loc["keybert", "soft_precision"])))
rows.append(("Sec. IV-A3 text", "KeyBERT soft recall vs LLM (0.203 against 0.467)", "0.203", float(means.loc["keybert", "soft_recall"])))
rows.append(("Sec. V-B", "LLM soft recall (five keywords cover 47% of expert keyphrases)", "47%", float(means.loc["llama_3.1_8b", "soft_recall"])))
rows.append(("Sec. V-B", "LLM soft precision (at 79% precision)", "79%", float(means.loc["llama_3.1_8b", "soft_precision"])))
check = pd.DataFrame(rows, columns=["location", "quantity", "main.tex", "computed"])
check["computed (rounded)"] = [fmt_like(m, v) for m, v in zip(check["main.tex"], check["computed"])]
check["flag"] = np.where(check["computed (rounded)"] == check["main.tex"], "match", "differs")
check["computed"] = [f"{v:.6g}" for v in check["computed"]]
print(check.to_string(index=False))
print(f"\n{(check.flag == 'match').sum()} of {len(check)} values match; {(check.flag == 'differs').sum()} differ.")

            location                                                       quantity main.tex  computed computed (rounded)  flag
Table 4 (Sec. IV-A3)                                          LLaMA 3.1 8B: Jaccard    0.142  0.142228              0.142 match
Table 4 (Sec. IV-A3)                                           LLaMA 3.1 8B: Soft P    0.794    0.7939              0.794 match
Table 4 (Sec. IV-A3)                                           LLaMA 3.1 8B: Soft R    0.467  0.466999              0.467 match
Table 4 (Sec. IV-A3)                                          LLaMA 3.1 8B: Soft F1    0.565  0.565183              0.565 match
Table 4 (Sec. IV-A3)                                              LLaMA 3.1 8B: SMM    0.752  0.752155              0.752 match
Table 4 (Sec. IV-A3)                                           LLaMA 3.1 8B: Global    0.804  0.804198              0.804 match
Table 4 (Sec. IV-A3)                                                TF-IDF: Jaccard    0.047 0.0469393  